# Morton Coverage of Linestrings — Antarctic InSAR Grounding Line

`mortie.linestring_coverage()` rasterizes an open polyline (or list of polylines) into a contiguous chain of HEALPix cells at a given order, returning morton indices.

**Return-shape contract**
- Single linestring → 1-D `np.ndarray` (sorted, unique).
- Multi-linestring → `list` of 1-D arrays, one per line (lengths may differ; not deduplicated across lines).

**Algorithm** (reused from the polygon-coverage Phase A). For each consecutive vertex pair, sample the great-circle arc at half-cell-resolution spacing and rasterize each sample. The result is a connected cell chain at the requested order (default `order=18`).

This notebook:
1. Demonstrates the API on a synthetic linestring.
2. Queries **NSIDC MEaSUREs InSAR Grounding Line v2.1** via `earthaccess`, loads the geopackage with `geopandas`, and unpacks the shapely `MultiLineString` into lat/lon arrays for `linestring_coverage`.
3. Plots the morton cells in a south-polar-stereographic projection.
4. Demonstrates two buffer strategies around the line:
   - `morton_buffer(cells, k=N)` — the existing k-cell-ring buffer from PR #17.
   - `morton_buffer_meters(cells, width_m=…)` — a convenience wrapper that picks `k` from a metric width (rounded up to the nearest whole cell width).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import healpy as hp
from shapely.geometry import Polygon as ShapelyPolygon
from shapely.geometry import LineString, MultiLineString

import mortie
print('mortie version:', mortie.__version__)

## Rendering helpers

Reused/adapted from `morton_coverage_example.ipynb`. Converts morton cells to shapely polygons for plotting, and splits linestrings that cross the antimeridian.

In [ ]:
def morton_cell_geometries(morton_indices, step=16):
    """Convert morton indices to shapely polygons in lon/lat coords.

    `step` controls boundary resolution: step=16 gives 64 points per cell,
    which traces curved HEALPix edges (important near the poles).
    """
    nested, order = mortie.mort2healpix(np.asarray(morton_indices))
    nested = np.atleast_1d(nested)
    nside = 2 ** order

    geoms = []
    for pix in nested:
        xyz = hp.boundaries(nside, int(pix), step=step, nest=True)
        lons, lats = hp.vec2ang(xyz.T, lonlat=True)
        center_lon, _ = hp.pix2ang(nside, int(pix), nest=True, lonlat=True)
        if center_lon > 180:
            center_lon -= 360
        lons = np.where(lons > 180, lons - 360, lons)
        # Antimeridian normalisation — shift vertices onto the same side as centre
        if center_lon < 0:
            lons = np.where(lons > 90, lons - 360, lons)
        else:
            lons = np.where(lons < -90, lons + 360, lons)
        geoms.append(ShapelyPolygon(zip(lons, lats)))
    return geoms


def plot_cells(ax, morton_indices, facecolor='#4a90e2', edgecolor='black',
               alpha=0.45, linewidth=0.2, step=8):
    """Render a set of morton cells on a cartopy axis."""
    if len(morton_indices) == 0:
        return
    geoms = morton_cell_geometries(morton_indices, step=step)
    ax.add_geometries(
        geoms, crs=ccrs.PlateCarree(),
        facecolor=facecolor, edgecolor=edgecolor,
        alpha=alpha, linewidth=linewidth,
    )


def plot_line(ax, lats, lons, color='crimson', lw=1.2):
    """Plot a lat/lon polyline on a cartopy axis (splits at antimeridian)."""
    lons = np.asarray(lons)
    lats = np.asarray(lats)
    dlon = np.abs(np.diff(lons))
    breaks = np.where(dlon > 180)[0]
    start = 0
    for b in list(breaks) + [len(lons) - 1]:
        sl = slice(start, b + 1)
        ax.plot(lons[sl], lats[sl], color=color, lw=lw, transform=ccrs.PlateCarree())
        start = b + 1

## 1. Synthetic example — a single linestring

A short polyline across the North Atlantic. We sweep three HEALPix orders so you can see the cell-size/coverage tradeoff.

In [ ]:
lats = np.array([40.0, 50.0, 55.0, 60.0])
lons = np.array([-70.0, -40.0, -20.0, 10.0])

fig = plt.figure(figsize=(18, 5))
proj = ccrs.Orthographic(central_longitude=-20, central_latitude=50)
for i, order in enumerate([4, 6, 8]):
    ax = fig.add_subplot(1, 3, i + 1, projection=proj)
    ax.set_global()
    ax.coastlines(resolution='110m', lw=0.4)
    ax.add_feature(cfeature.LAND, facecolor='#eaeaea')
    cells = mortie.linestring_coverage(lats, lons, order=order)
    plot_cells(ax, cells, step=4)
    plot_line(ax, lats, lons)
    ax.set_title(f'order {order} — {len(cells)} cells')
plt.tight_layout()
plt.show()

## 2. Real data — NSIDC InSAR Grounding Line (Antarctica)

`NSIDC-0498` (MEaSUREs Antarctic Grounding Line from Differential Satellite Radar Interferometry, v2.1) ships as a **geopackage** containing polyline features for each acquisition. We:

1. Use `earthaccess` to search CMR and download one granule.
2. Load with `geopandas.read_file` (GeoPackage via pyogrio).
3. Unpack shapely `LineString` / `MultiLineString` geometries into lat/lon arrays.
4. Call `mortie.linestring_coverage(...)` at order 10 (~10 km cells) for the whole file.

> **First run** will prompt for NASA Earthdata credentials via `earthaccess.login(persist=True)` (stored in `~/.netrc`). Subsequent runs reuse the stored credentials.

In [ ]:
import earthaccess
from pathlib import Path

auth = earthaccess.login(persist=True)

# Search for the InSAR_GL_Antarctica dataset, v2.1 — try the usual short_name
# candidates; the first match wins.
candidate_short_names = [
    'NSIDC-0498',             # MEaSUREs Antarctic Grounding Line
    'InSAR_GL_Antarctica',    # dataset alias
    'InSAR_GL',               # short alias
]
results = []
for sn in candidate_short_names:
    results = earthaccess.search_data(short_name=sn, version='2.1', count=5)
    if results:
        print(f'Using short_name={sn!r}, {len(results)} granules')
        break
else:
    raise RuntimeError('No granules found; check short_name / version in CMR')

# Download one granule to a local cache
cache = Path('./insar_gl_cache')
cache.mkdir(exist_ok=True)
files = earthaccess.download(results[:1], local_path=str(cache))
print('Downloaded:', files)

In [ ]:
import geopandas as gpd

# Pick the .gpkg file(s) from the downloaded set
gpkg_files = [Path(f) for f in files if str(f).lower().endswith('.gpkg')]
if not gpkg_files:
    # Some granules bundle multiple assets — search the cache recursively
    gpkg_files = list(cache.rglob('*.gpkg'))
assert gpkg_files, f'No .gpkg asset found in downloaded granule: {files}'

gdf = gpd.read_file(gpkg_files[0])
print(f'loaded {len(gdf)} features from {gpkg_files[0].name}')
print('CRS:', gdf.crs)
print('geometry types:', gdf.geom_type.value_counts().to_dict())

# Reproject to EPSG:4326 (lon/lat in degrees) if needed
if gdf.crs and gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(4326)
gdf.head()

### Unpacking shapely geometries → lat/lon arrays

`linestring_coverage` accepts plain lat/lon arrays (to avoid adding a shapely dependency to `mortie`). The helper below converts each shapely `LineString` / `MultiLineString` into the `(lats, lons)` pair (or list-of-pairs) the function expects.

In [ ]:
def shapely_to_latlon(geom):
    """Unpack a shapely LineString or MultiLineString into arrays.

    Returns
    -------
    (lats, lons) for a LineString (pair of 1-D arrays),
    (list_of_lats, list_of_lons) for a MultiLineString.
    """
    if isinstance(geom, LineString):
        xs, ys = np.asarray(geom.xy[0]), np.asarray(geom.xy[1])
        return ys, xs  # shapely xy = (lon, lat) → return (lats, lons)
    if isinstance(geom, MultiLineString):
        lats_parts, lons_parts = [], []
        for ls in geom.geoms:
            xs, ys = np.asarray(ls.xy[0]), np.asarray(ls.xy[1])
            lats_parts.append(ys); lons_parts.append(xs)
        return lats_parts, lons_parts
    raise TypeError(f'expected LineString or MultiLineString, got {type(geom).__name__}')


# Collect all vertices across every feature into ONE multi-linestring input —
# i.e., a list of (lats, lons) pairs where each pair is one polyline.
lats_parts, lons_parts = [], []
for geom in gdf.geometry:
    if geom is None or geom.is_empty:
        continue
    la, lo = shapely_to_latlon(geom)
    if isinstance(la, list):
        lats_parts.extend(la); lons_parts.extend(lo)
    else:
        lats_parts.append(la); lons_parts.append(lo)

print(f'{len(lats_parts)} linestrings, {sum(len(p) for p in lats_parts):,} total vertices')

In [ ]:
# Coverage at order 10 (~10 km) — per-line list of arrays
import time
t0 = time.perf_counter()
per_line = mortie.linestring_coverage(lats_parts, lons_parts, order=10)
dt = (time.perf_counter() - t0) * 1000
print(f'{len(per_line)} per-line arrays in {dt:.1f} ms')
print(f'cell counts: min={min(len(a) for a in per_line)}, '
      f'median={int(np.median([len(a) for a in per_line]))}, '
      f'max={max(len(a) for a in per_line)}')

# Union across all lines for plotting
all_cells = np.unique(np.concatenate(per_line))
print(f'union across all lines: {len(all_cells):,} cells')

In [ ]:
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.SouthPolarStereo())
ax.set_extent([-180, 180, -90, -60], ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, facecolor='#f1efe8')
ax.coastlines(resolution='50m', lw=0.3, color='#888')
ax.gridlines(draw_labels=False, lw=0.2, color='#ccc')

# Coverage cells in blue
plot_cells(ax, all_cells, facecolor='#4a90e2', edgecolor='#2858a0',
           alpha=0.6, linewidth=0.1, step=4)

# Overlay the raw linestrings in red
for la, lo in zip(lats_parts, lons_parts):
    plot_line(ax, la, lo, color='crimson', lw=0.5)

ax.set_title(f'InSAR Grounding Line — order 10 morton coverage ({len(all_cells):,} cells)')
plt.tight_layout()
plt.show()

## 3. Buffering the coverage

Two APIs:

| function | input | semantics |
|---|---|---|
| `morton_buffer(cells, k=N)` | integer `k` (cells) | returns the N-cell ring around the input (exclusive) |
| `morton_buffer_meters(cells, width_m=W)` | metric width in meters | picks `k = ceil(W / cell_width)` at the input order, then calls `morton_buffer`. **Approximate** — result is rounded UP to the nearest whole cell width, so the buffer covers *at least* `W` meters. |

Both return only the **border ring**, not the input cells. Use `np.union1d(cells, border)` to get the filled buffer.

In [ ]:
# Use one line of the InSAR GL dataset (the longest one) as the demo subject
lengths = [len(a) for a in per_line]
idx = int(np.argmax(lengths))
line_cells = per_line[idx]
la, lo = lats_parts[idx], lons_parts[idx]
print(f'line #{idx}: {len(la)} vertices -> {len(line_cells)} cells at order 10')

# (a) k-based buffer: 1-cell and 3-cell rings
ring_k1 = mortie.morton_buffer(line_cells, k=1)
ring_k3 = mortie.morton_buffer(line_cells, k=3)

# (b) meter-based buffer: ~50 km
# at order 10, cell width is ~11.2 km; ceil(50/11.2) = 5
ring_50km = mortie.morton_buffer_meters(line_cells, width_m=50_000.0)

print(f'ring k=1: {len(ring_k1)} cells')
print(f'ring k=3: {len(ring_k3)} cells')
print(f'ring 50 km (meter-based): {len(ring_50km)} cells')

In [ ]:
# Visualise the three buffers side-by-side around the same line
fig = plt.figure(figsize=(18, 7))

for i, (name, ring) in enumerate([
    ('morton_buffer k=1', ring_k1),
    ('morton_buffer k=3', ring_k3),
    ('morton_buffer_meters ~50 km', ring_50km),
]):
    ax = fig.add_subplot(1, 3, i + 1, projection=ccrs.SouthPolarStereo())
    # Zoom to the line's bounding box with a little padding
    lat_min, lat_max = np.min(la) - 1.5, np.max(la) + 1.5
    lon_min, lon_max = np.min(lo) - 4, np.max(lo) + 4
    ax.set_extent([lon_min, lon_max, max(lat_min, -90), min(lat_max, -60)],
                  ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor='#f1efe8')
    ax.coastlines(resolution='50m', lw=0.3, color='#888')

    plot_cells(ax, line_cells, facecolor='#4a90e2', edgecolor='#2858a0',
               alpha=0.75, step=4)
    plot_cells(ax, ring, facecolor='#ffb347', edgecolor='#c07000',
               alpha=0.55, step=4)
    plot_line(ax, la, lo, color='crimson', lw=1.0)
    ax.set_title(f'{name}\nline {len(line_cells)} cells + ring {len(ring)} cells')

plt.tight_layout()
plt.show()

### Filled buffer (union pattern)

`morton_buffer` returns **only the ring**. Union with the original cells to get a filled buffer.

In [ ]:
filled_50km = np.union1d(line_cells, ring_50km)
print(f'filled 50km buffer: {len(filled_50km)} cells '
      f'(= {len(line_cells)} line + {len(ring_50km)} ring)')

fig = plt.figure(figsize=(9, 9))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.SouthPolarStereo())
lat_min, lat_max = np.min(la) - 1.5, np.max(la) + 1.5
lon_min, lon_max = np.min(lo) - 4, np.max(lo) + 4
ax.set_extent([lon_min, lon_max, max(lat_min, -90), min(lat_max, -60)],
              ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, facecolor='#f1efe8')
ax.coastlines(resolution='50m', lw=0.3, color='#888')

plot_cells(ax, filled_50km, facecolor='#ffb347', edgecolor='#c07000',
           alpha=0.45, step=4)
plot_cells(ax, line_cells, facecolor='#2858a0', edgecolor='#1a3a6e',
           alpha=0.85, step=4)
plot_line(ax, la, lo, color='crimson', lw=1.0)
ax.set_title('filled 50 km buffer (line = dark blue, ring = amber)')
plt.tight_layout()
plt.show()